# 08 — GRU Baseline (Part A) and DMoN K-Sweep (Part B)

**Reddit Opinion & Community Research — closing two more reviewer-flagged gaps**



In [1]:
import sys, importlib

def ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        get_ipython().system(f"pip install -q {pkg}")

for pkg, imp in [("torch", None), ("numpy", None), ("pandas", None), ("pyarrow", None),
                 ("scikit-learn", "sklearn"), ("scipy", None)]:
    ensure(pkg, imp)

print("Packages OK.")


Packages OK.


In [2]:
import os, json, pickle, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score
from scipy import stats

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = "/content/drive/MyDrive/PhD/reddit_opinion_research"
else:
    BASE_DIR = os.environ.get("REDDIT_RESEARCH_BASE_DIR", "./reddit_opinion_research")

with open(os.path.join(BASE_DIR, "processed_v2", "dataset", "dataset_contract.json"), "r", encoding="utf-8") as f:
    dataset_contract = json.load(f)
DATASET_DIR = dataset_contract["dataset_dir"]
MODELS_DIR = os.path.join(BASE_DIR, "processed_v2", "models")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


Mounted at /content/drive
Using device: cuda


# Part A — Tree-Indexed GRU Baseline

## Load dataset and the SSM ablation results 

In [3]:
def load_split(name):
    with open(os.path.join(DATASET_DIR, f"{name}_items.pkl"), "rb") as f:
        return pickle.load(f)

splits = {name: load_split(name) for name in ["train", "val", "test"]}
embeddings = np.load(dataset_contract["embeddings_path"]).astype("float32")
node_feat_matrix = np.load(os.path.join(DATASET_DIR, "node_feat_matrix.npy"))

with open(dataset_contract["vocab_path"], "r", encoding="utf-8") as f:
    vocab = json.load(f)
N_SENTIMENT, N_EMOTION = len(vocab["sentiment_to_idx"]), len(vocab["emotion_to_idx"])

with open(os.path.join(MODELS_DIR, "ablation_chain_vs_tree.json"), "r", encoding="utf-8") as f:
    ssm_results = json.load(f)
print("Loaded SSM ablation results from notebook 07:")
print(f"  SSM-tree sentiment:  {ssm_results['sentiment']['tree_mean']:.3f} +/- {ssm_results['sentiment']['tree_std']:.3f}")
print(f"  SSM-chain sentiment: {ssm_results['sentiment']['chain_mean']:.3f} +/- {ssm_results['sentiment']['chain_std']:.3f}")


Loaded SSM ablation results from notebook 07:
  SSM-tree sentiment:  0.476 +/- 0.004
  SSM-chain sentiment: 0.457 +/- 0.008


## Collate, batching, chain-index 

In [4]:
def make_collate_fn(embeddings, node_feat_matrix):
    emb_dim = embeddings.shape[1]
    n_node_feats = node_feat_matrix.shape[1]

    def collate(batch):
        B = len(batch)
        L = max(it["length"] for it in batch)
        emb_batch = np.zeros((B, L, emb_dim), dtype="float32")
        node_feat_batch = np.zeros((B, L, n_node_feats), dtype="float32")
        parent_batch = np.full((B, L), -1, dtype="int64")
        sentiment_batch = np.full((B, L), -1, dtype="int64")
        emotion_batch = np.full((B, L), -1, dtype="int64")
        mask = np.zeros((B, L), dtype="bool")
        for i, it in enumerate(batch):
            n = it["length"]
            emb_batch[i, :n] = embeddings[it["emb_idx"]]
            valid_author = it["author_idx"] >= 0
            node_feat_batch[i, :n][valid_author] = node_feat_matrix[it["author_idx"][valid_author]]
            parent_batch[i, :n] = it["parent_idx"]
            sentiment_batch[i, :n] = it["sentiment_idx"]
            emotion_batch[i, :n] = it["emotion_idx"]
            mask[i, :n] = True
        return {
            "embeddings": torch.from_numpy(emb_batch), "node_features": torch.from_numpy(node_feat_batch),
            "parent_idx": torch.from_numpy(parent_batch),
            "sentiment_idx": torch.from_numpy(sentiment_batch), "emotion_idx": torch.from_numpy(emotion_batch),
            "mask": torch.from_numpy(mask),
        }
    return collate

collate_fn = make_collate_fn(embeddings, node_feat_matrix)


def make_length_bucketed_batches(items, batch_size, rng):
    lengths = np.array([it["length"] for it in items])
    order = np.argsort(lengths)
    batches = [order[i:i + batch_size].tolist() for i in range(0, len(order), batch_size)]
    rng.shuffle(batches)
    return batches


def make_chain_idx(mask):
    B, L = mask.shape
    idx = torch.arange(L, device=mask.device).unsqueeze(0).expand(B, L).clone() - 1
    return torch.where(idx < 0, torch.full_like(idx, -1), idx)


## Tree-indexed GRU cell 

In [5]:
class TreeRecurrentGRULayer(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.gru_cell = nn.GRUCell(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, parent_idx, mask):
        B, L, D = x.shape
        batch_arange = torch.arange(B, device=x.device)
        h_history, y_list = [], []
        for t in range(L):
            valid_t = mask[:, t]
            p = parent_idx[:, t]
            has_parent = p >= 0
            if t == 0:
                h_prev = x.new_zeros(B, D)
            else:
                h_stack = torch.stack(h_history, dim=1)
                gathered = h_stack[batch_arange, p.clamp(min=0, max=t - 1)]
                h_prev = torch.where(has_parent.view(B, 1), gathered, x.new_zeros(B, D))
            h_t = self.gru_cell(x[:, t], h_prev)
            h_t = torch.where(valid_t.view(B, 1), h_t, torch.zeros_like(h_t))
            h_history.append(h_t)
            y_list.append(h_t)
        return self.norm(torch.stack(y_list, dim=1) + x)


class HODGRUBaseline(nn.Module):
    def __init__(self, emb_dim, node_feat_dim, d_model=128, num_layers=4, n_sentiment=3, n_emotion=5):
        super().__init__()
        self.input_proj = nn.Linear(emb_dim, d_model)
        self.node_feat_proj = nn.Linear(node_feat_dim, d_model)
        self.layers = nn.ModuleList([TreeRecurrentGRULayer(d_model) for _ in range(num_layers)])
        self.root_context = nn.Parameter(torch.zeros(d_model))
        self.sentiment_head = nn.Linear(d_model, n_sentiment)
        self.emotion_head = nn.Linear(d_model, n_emotion)

    def forward(self, batch):
        x = self.input_proj(batch["embeddings"]) + self.node_feat_proj(batch["node_features"])
        parent_idx, mask = batch["parent_idx"], batch["mask"]
        for layer in self.layers:
            x = layer(x, parent_idx, mask)
        B, L, D = x.shape
        has_parent = parent_idx >= 0
        p = parent_idx.clamp(min=0)
        context = torch.stack([x[b, p[b]] for b in range(B)], dim=0)
        context = torch.where(has_parent.unsqueeze(-1), context, self.root_context.view(1, 1, D))
        return self.sentiment_head(context), self.emotion_head(context)


## Loss, mode-aware batch prep, evaluation, training loop 

In [6]:
def compute_loss(sent_logits, emo_logits, forecastable, batch):
    sent_valid = forecastable & (batch["sentiment_idx"] >= 0)
    emo_valid = forecastable & (batch["emotion_idx"] >= 0)
    loss = 0.0
    if sent_valid.any():
        loss = loss + F.cross_entropy(sent_logits[sent_valid], batch["sentiment_idx"][sent_valid])
    if emo_valid.any():
        loss = loss + F.cross_entropy(emo_logits[emo_valid], batch["emotion_idx"][emo_valid])
    return loss, sent_valid, emo_valid


def prepare_model_batch(batch, mode):
    tree_parent_idx = batch["parent_idx"]
    forecastable = batch["mask"] & (tree_parent_idx >= 0)
    if mode == "chain":
        model_batch = dict(batch)
        model_batch["parent_idx"] = make_chain_idx(batch["mask"])
    else:
        model_batch = batch
    return model_batch, forecastable


@torch.no_grad()
def evaluate(model, items, mode, batch_size=32):
    model.eval()
    rng = np.random.default_rng(0)
    batches = make_length_bucketed_batches(items, batch_size, rng)
    all_sp, all_st, all_ep, all_et = [], [], [], []
    for batch_idx in batches:
        batch = collate_fn([items[i] for i in batch_idx])
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        model_batch, forecastable = prepare_model_batch(batch, mode)
        sl, el = model(model_batch)
        _, sv, ev = compute_loss(sl, el, forecastable, batch)
        all_sp.append(sl[sv].argmax(-1).cpu().numpy()); all_st.append(batch["sentiment_idx"][sv].cpu().numpy())
        all_ep.append(el[ev].argmax(-1).cpu().numpy()); all_et.append(batch["emotion_idx"][ev].cpu().numpy())
    sp, st = np.concatenate(all_sp), np.concatenate(all_st)
    ep, et = np.concatenate(all_ep), np.concatenate(all_et)
    return (f1_score(st, sp, average="macro", zero_division=0),
            f1_score(et, ep, average="macro", zero_division=0))


def run_gru_variant(mode, seed, batch_size=32, max_epochs=30, patience=5):
    torch.manual_seed(seed)
    model = HODGRUBaseline(
        emb_dim=embeddings.shape[1], node_feat_dim=node_feat_matrix.shape[1],
        n_sentiment=N_SENTIMENT, n_emotion=N_EMOTION,
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    rng = np.random.default_rng(seed)

    best_combined_f1, best_state, patience_ctr = -1.0, None, 0
    for epoch in range(max_epochs):
        model.train()
        batches = make_length_bucketed_batches(splits["train"], batch_size, rng)
        for batch_idx in batches:
            batch = collate_fn([splits["train"][i] for i in batch_idx])
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            model_batch, forecastable = prepare_model_batch(batch, mode)
            optimizer.zero_grad()
            sl, el = model(model_batch)
            loss, _, _ = compute_loss(sl, el, forecastable, batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        val_sent_f1, val_emo_f1 = evaluate(model, splits["val"], mode, batch_size)
        combined_f1 = (val_sent_f1 + val_emo_f1) / 2
        if combined_f1 > best_combined_f1:
            best_combined_f1, patience_ctr = combined_f1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                break
    model.load_state_dict(best_state)
    test_sent_f1, test_emo_f1 = evaluate(model, splits["test"], mode, batch_size)
    return test_sent_f1, test_emo_f1, epoch + 1


## Run 5 seeds x 2 conditions for the GRU baseline

In [7]:
SEEDS = [0, 1, 2, 3, 4]
gru_results = {"tree": {"sentiment": [], "emotion": []}, "chain": {"sentiment": [], "emotion": []}}

for seed in SEEDS:
    for mode in ["tree", "chain"]:
        t0 = time.time()
        test_sent_f1, test_emo_f1, n_epochs = run_gru_variant(mode, seed)
        gru_results[mode]["sentiment"].append(test_sent_f1)
        gru_results[mode]["emotion"].append(test_emo_f1)
        print(f"seed={seed} mode={mode:5s}: sentiment_f1={test_sent_f1:.3f}, emotion_f1={test_emo_f1:.3f}, "
              f"{n_epochs} epochs, {time.time()-t0:.0f}s")


seed=0 mode=tree : sentiment_f1=0.470, emotion_f1=0.327, 13 epochs, 357s
seed=0 mode=chain: sentiment_f1=0.453, emotion_f1=0.308, 6 epochs, 158s
seed=1 mode=tree : sentiment_f1=0.462, emotion_f1=0.316, 8 epochs, 210s
seed=1 mode=chain: sentiment_f1=0.440, emotion_f1=0.319, 19 epochs, 500s
seed=2 mode=tree : sentiment_f1=0.483, emotion_f1=0.324, 10 epochs, 264s
seed=2 mode=chain: sentiment_f1=0.467, emotion_f1=0.314, 14 epochs, 372s
seed=3 mode=tree : sentiment_f1=0.480, emotion_f1=0.317, 10 epochs, 264s
seed=3 mode=chain: sentiment_f1=0.452, emotion_f1=0.304, 14 epochs, 368s
seed=4 mode=tree : sentiment_f1=0.479, emotion_f1=0.329, 9 epochs, 236s
seed=4 mode=chain: sentiment_f1=0.470, emotion_f1=0.314, 9 epochs, 238s


## Aggregate and compare against the SSM results

In [8]:
gru_summary = {}
for task in ["sentiment", "emotion"]:
    tree_scores = np.array(gru_results["tree"][task])
    chain_scores = np.array(gru_results["chain"][task])
    t_stat, p_value = stats.ttest_rel(tree_scores, chain_scores)
    gru_summary[task] = {
        "tree_mean": float(tree_scores.mean()), "tree_std": float(tree_scores.std()),
        "chain_mean": float(chain_scores.mean()), "chain_std": float(chain_scores.std()),
        "tree_scores": tree_scores.tolist(), "chain_scores": chain_scores.tolist(),
        "gru_tree_vs_chain_t": float(t_stat), "gru_tree_vs_chain_p": float(p_value),
    }

    # Cross-architecture: SSM-tree vs GRU-tree, same seeds -> valid paired comparison
    ssm_tree_scores = np.array(ssm_results[task]["tree_scores"])
    t_stat2, p_value2 = stats.ttest_rel(ssm_tree_scores, tree_scores)
    gru_summary[task]["ssm_tree_vs_gru_tree_t"] = float(t_stat2)
    gru_summary[task]["ssm_tree_vs_gru_tree_p"] = float(p_value2)

    print(f"[{task}] GRU-tree:  {tree_scores.mean():.3f} +/- {tree_scores.std():.3f}")
    print(f"[{task}] GRU-chain: {chain_scores.mean():.3f} +/- {chain_scores.std():.3f}")
    print(f"[{task}] GRU tree-vs-chain: t={t_stat:.3f}, p={p_value:.4f}")
    print(f"[{task}] SSM-tree ({ssm_tree_scores.mean():.3f}) vs GRU-tree ({tree_scores.mean():.3f}): "
          f"t={t_stat2:.3f}, p={p_value2:.4f}\n")

with open(os.path.join(MODELS_DIR, "gru_baseline_results.json"), "w", encoding="utf-8") as f:
    json.dump(gru_summary, f, indent=2, ensure_ascii=False)
print("Saved:", os.path.join(MODELS_DIR, "gru_baseline_results.json"))


[sentiment] GRU-tree:  0.475 +/- 0.008
[sentiment] GRU-chain: 0.457 +/- 0.011
[sentiment] GRU tree-vs-chain: t=5.827, p=0.0043
[sentiment] SSM-tree (0.476) vs GRU-tree (0.475): t=0.257, p=0.8099

[emotion] GRU-tree:  0.323 +/- 0.005
[emotion] GRU-chain: 0.312 +/- 0.005
[emotion] GRU tree-vs-chain: t=2.897, p=0.0442
[emotion] SSM-tree (0.322) vs GRU-tree (0.323): t=-0.092, p=0.9310

Saved: /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/models/gru_baseline_results.json


# Part B — DMoN K-Sweep

## Load the already-trained Hetero backbone and extract user embeddings

In [9]:
class RelationTypedSSMLayer(nn.Module):
    def __init__(self, d_model, state_dim=16, n_relations=3):
        super().__init__()
        self.d_model, self.state_dim = d_model, state_dim
        self.relation_embed = nn.Embedding(n_relations, d_model)
        self.A_log = nn.Parameter(torch.randn(d_model, state_dim) * 0.1)
        self.W_delta = nn.Linear(d_model, d_model)
        self.W_B = nn.Linear(d_model, state_dim)
        self.W_C = nn.Linear(d_model, state_dim)
        self.D = nn.Parameter(torch.randn(d_model) * 0.1)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, parent_idx, mask, relation_idx):
        B, L, D = x.shape
        N = self.state_dim
        x_eff = x + self.relation_embed(relation_idx)
        A = -torch.exp(self.A_log)
        delta = F.softplus(self.W_delta(x_eff))
        Bt = self.W_B(x_eff)
        Ct = self.W_C(x_eff)
        Abar = torch.exp(delta.unsqueeze(-1) * A.view(1, 1, D, N))
        Bbar = delta.unsqueeze(-1) * Bt.unsqueeze(2)
        batch_arange = torch.arange(B, device=x.device)
        h_history, y_list = [], []
        for t in range(L):
            valid_t = mask[:, t]
            p = parent_idx[:, t]
            has_parent = p >= 0
            if t == 0:
                h_prev = x.new_zeros(B, D, N)
            else:
                h_stack = torch.stack(h_history, dim=1)
                gathered = h_stack[batch_arange, p.clamp(min=0, max=t - 1)]
                h_prev = torch.where(has_parent.view(B, 1, 1), gathered, x.new_zeros(B, D, N))
            h_t = Abar[:, t] * h_prev + Bbar[:, t] * x_eff[:, t].unsqueeze(-1)
            h_t = torch.where(valid_t.view(B, 1, 1), h_t, torch.zeros_like(h_t))
            h_history.append(h_t)
            y_t = (Ct[:, t].unsqueeze(1) * h_t).sum(-1) + self.D * x_eff[:, t]
            y_t = torch.where(valid_t.view(B, 1), y_t, torch.zeros_like(y_t))
            y_list.append(y_t)
        return self.norm(torch.stack(y_list, dim=1) + x)


class HODMambaHetero(nn.Module):
    def __init__(self, emb_dim, node_feat_dim, d_model=128, state_dim=16, num_layers=4,
                 n_sentiment=3, n_emotion=5, n_relations=3):
        super().__init__()
        self.input_proj = nn.Linear(emb_dim, d_model)
        self.node_feat_proj = nn.Linear(node_feat_dim, d_model)
        self.layers = nn.ModuleList([
            RelationTypedSSMLayer(d_model, state_dim, n_relations) for _ in range(num_layers)
        ])
        self.root_context = nn.Parameter(torch.zeros(d_model))
        self.sentiment_head = nn.Linear(d_model, n_sentiment)
        self.emotion_head = nn.Linear(d_model, n_emotion)

    def encode(self, batch):
        x = self.input_proj(batch["embeddings"]) + self.node_feat_proj(batch["node_features"])
        parent_idx, mask, relation_idx = batch["parent_idx"], batch["mask"], batch["relation_idx"]
        for layer in self.layers:
            x = layer(x, parent_idx, mask, relation_idx)
        return x

    def forward(self, batch):
        x = self.encode(batch)
        B, L, D = x.shape
        parent_idx = batch["parent_idx"]
        has_parent = parent_idx >= 0
        p = parent_idx.clamp(min=0)
        context = torch.stack([x[b, p[b]] for b in range(B)], dim=0)
        context = torch.where(has_parent.unsqueeze(-1), context, self.root_context.view(1, 1, D))
        return self.sentiment_head(context), self.emotion_head(context)


hetero_model = HODMambaHetero(
    emb_dim=embeddings.shape[1], node_feat_dim=node_feat_matrix.shape[1],
    n_sentiment=N_SENTIMENT, n_emotion=N_EMOTION,
).to(DEVICE)
hetero_model.load_state_dict(torch.load(os.path.join(MODELS_DIR, "hod_mamba_hetero.pt"), map_location=DEVICE))
hetero_model.eval()
print("Loaded the already-trained Hetero backbone -- no retraining needed for the K-sweep.")


Loaded the already-trained Hetero backbone -- no retraining needed for the K-sweep.


In [10]:
def make_collate_fn_hetero(embeddings, node_feat_matrix):
    emb_dim = embeddings.shape[1]
    n_node_feats = node_feat_matrix.shape[1]

    def collate(batch):
        B = len(batch)
        L = max(it["length"] for it in batch)
        emb_batch = np.zeros((B, L, emb_dim), dtype="float32")
        node_feat_batch = np.zeros((B, L, n_node_feats), dtype="float32")
        parent_batch = np.full((B, L), -1, dtype="int64")
        relation_batch = np.zeros((B, L), dtype="int64")
        author_batch = np.full((B, L), -1, dtype="int64")
        mask = np.zeros((B, L), dtype="bool")
        for i, it in enumerate(batch):
            n = it["length"]
            emb_batch[i, :n] = embeddings[it["emb_idx"]]
            valid_author = it["author_idx"] >= 0
            node_feat_batch[i, :n][valid_author] = node_feat_matrix[it["author_idx"][valid_author]]
            parent_batch[i, :n] = it["parent_idx"]
            relation_batch[i, :n] = it["relation_idx"]
            author_batch[i, :n] = it["author_idx"]
            mask[i, :n] = True
        return {
            "embeddings": torch.from_numpy(emb_batch), "node_features": torch.from_numpy(node_feat_batch),
            "parent_idx": torch.from_numpy(parent_batch), "relation_idx": torch.from_numpy(relation_batch),
            "author_idx": torch.from_numpy(author_batch), "mask": torch.from_numpy(mask),
        }
    return collate

collate_fn_hetero = make_collate_fn_hetero(embeddings, node_feat_matrix)
all_items = splits["train"] + splits["val"] + splits["test"]
N_AUTHORS = node_feat_matrix.shape[0]

@torch.no_grad()
def extract_user_embeddings(model, items, n_authors, d_model, batch_size=32):
    model.eval()
    sum_embed = torch.zeros(n_authors, d_model)
    count = torch.zeros(n_authors)
    rng = np.random.default_rng(0)
    batches = make_length_bucketed_batches(items, batch_size, rng)
    for batch_idx in batches:
        batch = collate_fn_hetero([items[i] for i in batch_idx])
        batch_gpu = {k: v.to(DEVICE) for k, v in batch.items()}
        reprs = model.encode(batch_gpu).cpu()
        mask, author_idx = batch["mask"], batch["author_idx"]
        valid = mask & (author_idx >= 0)
        sum_embed.index_add_(0, author_idx[valid], reprs[valid])
        count.index_add_(0, author_idx[valid], torch.ones(valid.sum().item()))
    count = count.clamp(min=1)
    return sum_embed / count.unsqueeze(1)

d_model = hetero_model.input_proj.out_features
user_embeddings = extract_user_embeddings(hetero_model, all_items, N_AUTHORS, d_model)
print("user_embeddings:", user_embeddings.shape)


user_embeddings: torch.Size([22465, 128])


## Rebuild the static graph and DMoN loss 

In [11]:
static_graph = pd.read_parquet(dataset_contract["static_graph_path"])
normalized_weight = static_graph["weight"].astype("float64").copy()
for rel in static_graph["relation"].unique():
    rel_mask = static_graph["relation"] == rel
    total = normalized_weight[rel_mask].sum()
    if total > 0:
        normalized_weight[rel_mask] = normalized_weight[rel_mask] / total
static_graph["weight_norm"] = normalized_weight

src = torch.tensor(static_graph["source_idx"].values, dtype=torch.long)
dst = torch.tensor(static_graph["target_idx"].values, dtype=torch.long)
w = torch.tensor(static_graph["weight_norm"].values, dtype=torch.float32)
src_sym = torch.cat([src, dst]).to(DEVICE)
dst_sym = torch.cat([dst, src]).to(DEVICE)
w_sym = torch.cat([w, w]).to(DEVICE)


class DMoNHead(nn.Module):
    def __init__(self, d_in, n_clusters):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(d_in, d_in), nn.ReLU(), nn.Linear(d_in, n_clusters))

    def forward(self, x):
        return torch.softmax(self.mlp(x), dim=-1)


def dmon_loss(C, src, dst, w):
    n, k = C.shape
    d = torch.zeros(n, device=C.device).index_add_(0, src, w)
    m2 = d.sum()
    edge_term = (w * (C[src] * C[dst]).sum(dim=1)).sum()
    Cd = (C * d.unsqueeze(1)).sum(dim=0)
    degree_term = (Cd @ Cd) / m2
    L_mod = -(1.0 / m2) * (edge_term - degree_term)
    collapse_reg = (torch.sqrt(torch.tensor(float(k), device=C.device)) / n) * torch.norm(C.sum(dim=0), p=2) - 1
    return L_mod, collapse_reg


@torch.no_grad()
def hard_modularity(cluster_ids, src, dst, w):
    n = cluster_ids.shape[0]
    d = torch.zeros(n, device=cluster_ids.device).index_add_(0, src, w)
    m2 = d.sum()
    same_cluster = (cluster_ids[src] == cluster_ids[dst]).float()
    edge_term = (w * same_cluster).sum()
    k = int(cluster_ids.max().item()) + 1
    cluster_degree_sum = torch.zeros(k, device=cluster_ids.device).index_add_(0, cluster_ids, d)
    degree_term = (cluster_degree_sum ** 2).sum() / m2
    return ((edge_term - degree_term) / m2).item()

dmon_input = torch.cat([user_embeddings, torch.from_numpy(node_feat_matrix)], dim=1).to(DEVICE)
print("dmon_input:", dmon_input.shape)


dmon_input: torch.Size([22465, 134])


## Sweep K

In [12]:
K_VALUES = [4, 6, 8, 10, 12, 15, 20, 25, 30]
DEGENERATE_THRESHOLD = 50  # a cluster below this size is too small to interpret meaningfully at this scale

sweep_results = []
for K in K_VALUES:
    torch.manual_seed(42)
    dmon_head = DMoNHead(dmon_input.shape[1], K).to(DEVICE)
    optimizer = torch.optim.Adam(dmon_head.parameters(), lr=1e-2, weight_decay=1e-4)
    for epoch in range(300):
        optimizer.zero_grad()
        C = dmon_head(dmon_input)
        L_mod, L_collapse = dmon_loss(C, src_sym, dst_sym, w_sym)
        loss = L_mod + L_collapse
        loss.backward()
        optimizer.step()

    cluster_ids = dmon_head(dmon_input).argmax(dim=1)
    q = hard_modularity(cluster_ids, src_sym, dst_sym, w_sym)
    sizes = torch.bincount(cluster_ids, minlength=K).cpu().numpy()
    n_degenerate = int((sizes < DEGENERATE_THRESHOLD).sum())
    sweep_results.append({"K": K, "Q": float(q), "cluster_sizes": sorted(sizes.tolist()), "n_degenerate": n_degenerate})
    print(f"K={K:3d}: Q={q:.4f}, smallest/largest cluster={min(sizes)}/{max(sizes)}, "
          f"degenerate (<{DEGENERATE_THRESHOLD})={n_degenerate}")

with open(os.path.join(MODELS_DIR, "dmon_k_sweep.json"), "w", encoding="utf-8") as f:
    json.dump({"degenerate_threshold": DEGENERATE_THRESHOLD, "sweep": sweep_results}, f, indent=2, ensure_ascii=False)
print("\nSaved:", os.path.join(MODELS_DIR, "dmon_k_sweep.json"))


K=  4: Q=0.1537, smallest/largest cluster=5410/5918, degenerate (<50)=0
K=  6: Q=0.1398, smallest/largest cluster=3349/3989, degenerate (<50)=0
K=  8: Q=0.1293, smallest/largest cluster=2300/3055, degenerate (<50)=0
K= 10: Q=0.1214, smallest/largest cluster=1007/2861, degenerate (<50)=0
K= 12: Q=0.1210, smallest/largest cluster=945/2471, degenerate (<50)=0
K= 15: Q=0.1058, smallest/largest cluster=604/2420, degenerate (<50)=0
K= 20: Q=0.0971, smallest/largest cluster=20/2697, degenerate (<50)=1
K= 25: Q=0.0882, smallest/largest cluster=109/1559, degenerate (<50)=0
K= 30: Q=0.0222, smallest/largest cluster=0/4231, degenerate (<50)=16

Saved: /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/models/dmon_k_sweep.json
